In [1]:
import json
from pathlib import Path

import pandas as pd


ROOT_DIR = Path("ctg_full_2005_2025")

INPUT_FILE = (
    ROOT_DIR
    / "preprocessing"
    / "eligibility_features.csv"
)

OUTPUT_DIR = (
    ROOT_DIR
    / "preprocessing_qc"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXPECTED_START_YEAR_MIN = 2005
EXPECTED_START_YEAR_MAX = 2025

In [2]:
if not INPUT_FILE.exists():
    raise RuntimeError(
        f"Preprocessing output not found: {INPUT_FILE}"
    )


df = pd.read_csv(
    INPUT_FILE,
    encoding="utf-8-sig",
)


domain_indicator_columns = [
    "domain_neoplasms",
    "domain_nervous_system_diseases",
    "domain_cardiovascular_diseases",
    "domain_metabolic_diseases",
    "domain_respiratory_tract_diseases",
    "domain_infections",
]


required_columns = [
    "nct_id",
    "study_start_year",

    "disease_domains",
    "n_disease_domains",
    "disease_classification_status",
    *domain_indicator_columns,

    "has_inclusion_section",
    "has_exclusion_section",
    "section_split_success",
    "list_structure_detected",
    "listed_item_count_available",

    "n_inclusion_listed_items",
    "n_exclusion_listed_items",
    "n_total_listed_items",

    "inclusion_word_count",
    "exclusion_word_count",
    "total_eligibility_word_count",
    "sentence_count",
    "mean_sentence_length",
    "flesch_reading_ease",

    "quantitative_count",
    "quantitative_present",
    "quantitative_rate_per_1000_words",

    "temporal_count",
    "temporal_present",
    "temporal_rate_per_1000_words",

    "negation_count",
    "negation_present",
    "negation_rate_per_1000_words",

    "conditional_count",
    "conditional_present",
    "conditional_rate_per_1000_words",

    "exception_count",
    "exception_present",
    "exception_rate_per_1000_words",

    "preprocessing_warning",
]


missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise RuntimeError(
        "Required preprocessing-QC columns are missing: "
        f"{missing_columns}"
    )


print(f"Preprocessing dataset loaded: {len(df)} studies")
print(f"Columns available: {len(df.columns)}")

Preprocessing dataset loaded: 197671 studies
Columns available: 60


In [3]:
n_records = len(df)

n_unique_ids = df["nct_id"].nunique(
    dropna=True
)

n_missing_ids = int(
    df["nct_id"].isna().sum()
)

n_duplicate_id_rows = int(
    df["nct_id"]
    .duplicated(keep=False)
    .sum()
)


if n_missing_ids != 0:
    raise RuntimeError(
        f"Missing NCT IDs found: {n_missing_ids}"
    )

if n_duplicate_id_rows != 0:
    raise RuntimeError(
        f"Duplicate NCT-ID rows found: {n_duplicate_id_rows}"
    )

if n_unique_ids != n_records:
    raise RuntimeError(
        "Unique NCT-ID count does not match the number "
        "of preprocessing records."
    )


start_year = pd.to_numeric(
    df["study_start_year"],
    errors="coerce",
)

n_missing_start_year = int(
    start_year.isna().sum()
)

n_out_of_range_start_year = int(
    (
        start_year.notna()
        & (
            (start_year < EXPECTED_START_YEAR_MIN)
            | (start_year > EXPECTED_START_YEAR_MAX)
        )
    ).sum()
)


if n_missing_start_year != 0:
    raise RuntimeError(
        f"Missing Study Start Year values found: "
        f"{n_missing_start_year}"
    )

if n_out_of_range_start_year != 0:
    raise RuntimeError(
        "Study Start Year values outside the expected "
        f"{EXPECTED_START_YEAR_MIN}–"
        f"{EXPECTED_START_YEAR_MAX} range found: "
        f"{n_out_of_range_start_year}"
    )


# Every retained study must belong to at least one selected
# disease domain.
if df["n_disease_domains"].isna().any():
    raise RuntimeError(
        "Missing n_disease_domains values found."
    )

if (
    df["n_disease_domains"] < 1
).any():
    raise RuntimeError(
        "Preprocessing dataset contains studies with no "
        "selected disease-domain membership."
    )


# Only single-domain and multiple-domain studies should be present.
valid_classification_statuses = {
    "single_domain",
    "multiple_domains",
}

if not df[
    "disease_classification_status"
].isin(valid_classification_statuses).all():
    raise RuntimeError(
        "Unexpected disease-classification status found "
        "in the preprocessing dataset."
    )


# Disease-domain indicators must reconcile with
# n_disease_domains.
indicator_domain_count = (
    df[domain_indicator_columns]
    .sum(axis=1)
)

if not (
    indicator_domain_count
    == df["n_disease_domains"]
).all():
    raise RuntimeError(
        "Disease-domain indicators do not reconcile with "
        "n_disease_domains."
    )


# Every prespecified disease domain must be represented.
unrepresented_domains = [
    column
    for column in domain_indicator_columns
    if not df[column].any()
]

if unrepresented_domains:
    raise RuntimeError(
        "One or more prespecified disease domains are not "
        f"represented: {unrepresented_domains}"
    )


n_single_domain = int(
    (
        df["n_disease_domains"] == 1
    ).sum()
)

n_multiple_domains = int(
    (
        df["n_disease_domains"] > 1
    ).sum()
)


print(
    "Cohort identity, year-range, and disease-domain "
    "checks passed."
)
print(f"Rows: {n_records}")
print(f"Unique NCT IDs: {n_unique_ids}")
print(
    f"Study Start Year range: "
    f"{int(start_year.min())}–{int(start_year.max())}"
)
print(
    "Studies with single-domain membership: "
    f"{n_single_domain}"
)
print(
    "Studies with multiple-domain membership: "
    f"{n_multiple_domains}"
)
print(
    "Disease domains represented: "
    f"{sum(df[column].any() for column in domain_indicator_columns)}"
)

Cohort identity, year-range, and disease-domain checks passed.
Rows: 197671
Unique NCT IDs: 197671
Study Start Year range: 2005–2025
Studies with single-domain membership: 153677
Studies with multiple-domain membership: 43994
Disease domains represented: 6


In [4]:
structural_columns = [
    "n_inclusion_listed_items",
    "n_exclusion_listed_items",
    "n_total_listed_items",
    "inclusion_word_count",
    "exclusion_word_count",
]

whole_field_columns = [
    "total_eligibility_word_count",
    "sentence_count",
    "mean_sentence_length",
    "flesch_reading_ease",
    "quantitative_count",
    "temporal_count",
    "negation_count",
    "conditional_count",
    "exception_count",
]


# Whole-field measures must be complete for every retained study.
for column in whole_field_columns:
    n_missing = int(df[column].isna().sum())

    if n_missing != 0:
        raise RuntimeError(
            f"Unexpected missing values found in {column}: "
            f"{n_missing}"
        )


# Structural measures may legitimately be unavailable.
structural_missingness = pd.DataFrame(
    {
        "variable": structural_columns,
        "n_missing": [
            int(df[column].isna().sum())
            for column in structural_columns
        ],
    }
)

structural_missingness["percent_missing"] = (
    structural_missingness["n_missing"]
    / len(df)
    * 100
)


# Reconcile availability indicators with the corresponding variables.
expected_listed_availability = (
    df["n_total_listed_items"].notna()
)

if not (
    df["listed_item_count_available"]
    == expected_listed_availability
).all():
    raise RuntimeError(
        "listed_item_count_available is inconsistent with "
        "n_total_listed_items."
    )


n_no_section_heading = int(
    (~df["section_split_success"]).sum()
)

n_no_list_structure = int(
    (~df["listed_item_count_available"]).sum()
)


print("Missingness and structural-availability checks passed.")
print(
    f"Studies without recognized section headings: "
    f"{n_no_section_heading}"
)
print(
    f"Studies without available listed-item counts: "
    f"{n_no_list_structure}"
)
print()
print(
    structural_missingness.to_string(
        index=False,
        formatters={
            "percent_missing": "{:.2f}".format
        },
    )
)

Missingness and structural-availability checks passed.
Studies without recognized section headings: 3029
Studies without available listed-item counts: 2895

                variable  n_missing percent_missing
n_inclusion_listed_items       7934            4.01
n_exclusion_listed_items      10235            5.18
    n_total_listed_items       2895            1.46
    inclusion_word_count       3772            1.91
    exclusion_word_count       5475            2.77


In [5]:
semantic_characteristic_names = [
    "quantitative",
    "temporal",
    "negation",
    "conditional",
    "exception",
]


# Count variables must be non-negative.
count_columns = [
    "n_inclusion_listed_items",
    "n_exclusion_listed_items",
    "n_total_listed_items",
    "inclusion_word_count",
    "exclusion_word_count",
    "total_eligibility_word_count",
    "sentence_count",
    "quantitative_count",
    "temporal_count",
    "negation_count",
    "conditional_count",
    "exception_count",
]

for column in count_columns:
    invalid_mask = (
        df[column].notna()
        & (df[column] < 0)
    )

    if invalid_mask.any():
        raise RuntimeError(
            f"Negative values found in {column}: "
            f"{int(invalid_mask.sum())}"
        )


# Whole-field word and sentence counts must be positive.
if (df["total_eligibility_word_count"] <= 0).any():
    raise RuntimeError(
        "Zero or negative total eligibility word counts found."
    )

if (df["sentence_count"] <= 0).any():
    raise RuntimeError(
        "Zero or negative sentence counts found."
    )


# Mean sentence length must reconcile exactly with
# total words divided by sentence-like units.
expected_mean_sentence_length = (
    df["total_eligibility_word_count"]
    / df["sentence_count"]
)

mean_sentence_difference = (
    df["mean_sentence_length"]
    - expected_mean_sentence_length
).abs()

if (mean_sentence_difference > 1e-10).any():
    raise RuntimeError(
        "Mean sentence length reconciliation failed."
    )


# Section-split success must mean that at least one
# recognized section is present.
expected_split_success = (
    df["has_inclusion_section"]
    | df["has_exclusion_section"]
)

if not (
    df["section_split_success"]
    == expected_split_success
).all():
    raise RuntimeError(
        "section_split_success is inconsistent with "
        "the inclusion/exclusion section indicators."
    )


# Semantic-characteristic counts, presence indicators,
# and normalized rates must be mutually consistent.
for characteristic_name in semantic_characteristic_names:

    count_column = f"{characteristic_name}_count"
    present_column = f"{characteristic_name}_present"
    rate_column = (
        f"{characteristic_name}_rate_per_1000_words"
    )

    if df[count_column].isna().any():
        raise RuntimeError(
            f"Missing values found in {count_column}."
        )

    if df[present_column].isna().any():
        raise RuntimeError(
            f"Missing values found in {present_column}."
        )

    if df[rate_column].isna().any():
        raise RuntimeError(
            f"Missing values found in {rate_column}."
        )

    expected_present = (
        df[count_column] > 0
    )

    if not (
        df[present_column]
        == expected_present
    ).all():
        raise RuntimeError(
            f"{present_column} is inconsistent with "
            f"{count_column}."
        )

    expected_rate = (
        df[count_column]
        / df["total_eligibility_word_count"]
        * 1000
    )

    rate_difference = (
        df[rate_column]
        - expected_rate
    ).abs()

    if (rate_difference > 1e-10).any():
        raise RuntimeError(
            f"{rate_column} reconciliation failed."
        )


print("Logical consistency checks passed.")

Logical consistency checks passed.


In [6]:
distribution_columns = [
    "n_total_listed_items",
    "total_eligibility_word_count",
    "sentence_count",
    "mean_sentence_length",
    "flesch_reading_ease",

    "quantitative_count",
    "quantitative_rate_per_1000_words",

    "temporal_count",
    "temporal_rate_per_1000_words",

    "negation_count",
    "negation_rate_per_1000_words",

    "conditional_count",
    "conditional_rate_per_1000_words",

    "exception_count",
    "exception_rate_per_1000_words",
]


distribution_summary = (
    df[distribution_columns]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
    .T
    .reset_index()
    .rename(columns={"index": "variable"})
)


print("Distribution summary")
print(
    distribution_summary.to_string(
        index=False
    )
)


distribution_summary = (
    df[distribution_columns]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
    .T
    .reset_index()
    .rename(columns={"index": "variable"})
)


print("Distribution summary")
print(
    distribution_summary.to_string(
        index=False
    )
)

Distribution summary
                        variable    count       mean        std       min        1%      5%       25%        50%        75%        95%         99%         max
            n_total_listed_items 194776.0  15.063776   9.894901    1.0000  2.000000  3.0000  8.000000  13.000000  20.000000  34.000000   46.000000  147.000000
    total_eligibility_word_count 197671.0 269.878444 300.069355    1.0000 13.000000 30.0000 78.000000 157.000000 349.000000 891.000000 1436.000000 2978.000000
                  sentence_count 197671.0  18.746953  15.425668    1.0000  2.000000  3.0000  9.000000  14.000000  24.000000  49.000000   75.000000  242.000000
            mean_sentence_length 197671.0  13.036762   7.015918    1.0000  3.375000  5.0000  8.454545  12.000000  16.333333  23.482759   33.969565  214.750000
             flesch_reading_ease 197671.0  35.708456  14.177363 -301.2725 -5.181154 14.2675 28.353420  35.576118  43.481408  58.054187   70.630000  121.220000
              quantitativ

In [7]:
warning_series = (
    df["preprocessing_warning"]
    .fillna("")
    .astype(str)
    .str.strip()
)


expected_warning = pd.Series(
    "",
    index=df.index,
    dtype="object",
)

no_section = (
    ~df["section_split_success"]
)

no_list = (
    ~df["listed_item_count_available"]
)

expected_warning.loc[
    no_section & ~no_list
] = "NO_SECTION_HEADING"

expected_warning.loc[
    ~no_section & no_list
] = "NO_LIST_STRUCTURE"

expected_warning.loc[
    no_section & no_list
] = "NO_SECTION_HEADING|NO_LIST_STRUCTURE"


warning_mismatch = (
    warning_series != expected_warning
)

if warning_mismatch.any():
    raise RuntimeError(
        "preprocessing_warning is inconsistent with "
        "the structural availability indicators for "
        f"{int(warning_mismatch.sum())} studies."
    )


warning_summary = (
    warning_series
    .replace("", "NONE")
    .value_counts()
    .rename_axis("preprocessing_warning")
    .reset_index(name="n_studies")
)


print("Preprocessing-warning reconciliation passed.")
print()
print(
    warning_summary.to_string(
        index=False
    )
)

Preprocessing-warning reconciliation passed.

               preprocessing_warning  n_studies
                                NONE     191906
                  NO_SECTION_HEADING       2870
                   NO_LIST_STRUCTURE       2736
NO_SECTION_HEADING|NO_LIST_STRUCTURE        159


In [8]:
qc_summary = {
    "input_file": str(INPUT_FILE),

    "cohort_integrity": {
        "n_studies": int(len(df)),
        "n_unique_nct_ids": int(
            df["nct_id"].nunique()
        ),
        "n_missing_nct_ids": int(
            df["nct_id"].isna().sum()
        ),
        "n_duplicate_nct_id_rows": int(
            df["nct_id"]
            .duplicated(keep=False)
            .sum()
        ),
        "start_year_min": int(
            start_year.min()
        ),
        "start_year_max": int(
            start_year.max()
        ),
    },

    "disease_domains": {
        "Neoplasms": int(
            df["domain_neoplasms"].sum()
        ),
        "Nervous System Diseases": int(
            df[
                "domain_nervous_system_diseases"
            ].sum()
        ),
        "Cardiovascular Diseases": int(
            df[
                "domain_cardiovascular_diseases"
            ].sum()
        ),
        "Metabolic Diseases": int(
            df[
                "domain_metabolic_diseases"
            ].sum()
        ),
        "Respiratory Tract Diseases": int(
            df[
                "domain_respiratory_tract_diseases"
            ].sum()
        ),
        "Infections": int(
            df["domain_infections"].sum()
        ),
    },

    "disease_domain_membership": {
        "single_domain": int(
            (
                df["n_disease_domains"] == 1
            ).sum()
        ),
        "multiple_domains": int(
            (
                df["n_disease_domains"] > 1
            ).sum()
        ),
    },

    "structural_availability": {
        "n_without_recognized_section_heading": int(
            (~df["section_split_success"]).sum()
        ),
        "n_without_available_listed_item_count": int(
            (~df["listed_item_count_available"]).sum()
        ),
    },

    "structural_missingness": {
        str(row["variable"]): {
            "n_missing": int(row["n_missing"]),
            "percent_missing": float(
                row["percent_missing"]
            ),
        }
        for _, row in structural_missingness.iterrows()
    },

    "preprocessing_warnings": {
        str(row["preprocessing_warning"]): int(
            row["n_studies"]
        )
        for _, row in warning_summary.iterrows()
    },

    "qc_results": {
        "cohort_identity_year_range_and_disease_domains": "PASS",
        "whole_field_missingness": "PASS",
        "structural_availability": "PASS",
        "logical_consistency": "PASS",
        "warning_reconciliation": "PASS",
        "distribution_review": "PASS",
    },

    "additional_exclusions_after_preprocessing_qc": 0,
}


QC_SUMMARY_FILE = (
    OUTPUT_DIR
    / "preprocessing_qc_summary.json"
)

with open(
    QC_SUMMARY_FILE,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        qc_summary,
        f,
        ensure_ascii=False,
        indent=2,
    )


print(
    f"Preprocessing QC summary saved to: "
    f"{QC_SUMMARY_FILE}"
)

Preprocessing QC summary saved to: ctg_full_2005_2025/preprocessing_qc/preprocessing_qc_summary.json


In [9]:
with open(
    QC_SUMMARY_FILE,
    "r",
    encoding="utf-8",
) as f:
    saved_qc_summary = json.load(f)


if (
    saved_qc_summary["cohort_integrity"]["n_studies"]
    != len(df)
):
    raise RuntimeError(
        "Saved QC summary contains an incorrect study count."
    )

if (
    saved_qc_summary["cohort_integrity"]["n_unique_nct_ids"]
    != df["nct_id"].nunique()
):
    raise RuntimeError(
        "Saved QC summary contains an incorrect "
        "unique NCT-ID count."
    )


expected_disease_domain_counts = {
    "Neoplasms": int(
        df["domain_neoplasms"].sum()
    ),
    "Nervous System Diseases": int(
        df["domain_nervous_system_diseases"].sum()
    ),
    "Cardiovascular Diseases": int(
        df["domain_cardiovascular_diseases"].sum()
    ),
    "Metabolic Diseases": int(
        df["domain_metabolic_diseases"].sum()
    ),
    "Respiratory Tract Diseases": int(
        df["domain_respiratory_tract_diseases"].sum()
    ),
    "Infections": int(
        df["domain_infections"].sum()
    ),
}

if (
    saved_qc_summary["disease_domains"]
    != expected_disease_domain_counts
):
    raise RuntimeError(
        "Saved QC summary contains incorrect "
        "disease-domain counts."
    )


expected_membership_counts = {
    "single_domain": int(
        (
            df["n_disease_domains"] == 1
        ).sum()
    ),
    "multiple_domains": int(
        (
            df["n_disease_domains"] > 1
        ).sum()
    ),
}

if (
    saved_qc_summary["disease_domain_membership"]
    != expected_membership_counts
):
    raise RuntimeError(
        "Saved QC summary contains incorrect "
        "disease-domain membership counts."
    )


if (
    saved_qc_summary[
        "additional_exclusions_after_preprocessing_qc"
    ]
    != 0
):
    raise RuntimeError(
        "Saved QC summary incorrectly reports "
        "additional QC exclusions."
    )


expected_qc_results = {
    "cohort_identity_year_range_and_disease_domains": "PASS",
    "whole_field_missingness": "PASS",
    "structural_availability": "PASS",
    "logical_consistency": "PASS",
    "warning_reconciliation": "PASS",
    "distribution_review": "PASS",
}

if (
    saved_qc_summary["qc_results"]
    != expected_qc_results
):
    raise RuntimeError(
        "Saved QC summary does not contain the "
        "expected QC results."
    )


print(
    "Saved preprocessing-QC summary passed "
    "read-back checks."
)
print(
    f"Studies: "
    f"{saved_qc_summary['cohort_integrity']['n_studies']}"
)
print(
    f"Unique NCT IDs: "
    f"{saved_qc_summary['cohort_integrity']['n_unique_nct_ids']}"
)
print(
    f"Disease domains represented: "
    f"{len(saved_qc_summary['disease_domains'])}"
)
print(
    "Studies with single-domain membership: "
    f"{saved_qc_summary['disease_domain_membership']['single_domain']}"
)
print(
    "Studies with multiple-domain membership: "
    f"{saved_qc_summary['disease_domain_membership']['multiple_domains']}"
)
print(
    "Additional exclusions after preprocessing QC: "
    f"{saved_qc_summary['additional_exclusions_after_preprocessing_qc']}"
)
print("All preprocessing-QC checks: PASS")

Saved preprocessing-QC summary passed read-back checks.
Studies: 197671
Unique NCT IDs: 197671
Disease domains represented: 6
Studies with single-domain membership: 153677
Studies with multiple-domain membership: 43994
Additional exclusions after preprocessing QC: 0
All preprocessing-QC checks: PASS
